## IMPORTS


In [31]:
import pandas as pd
import numpy as np
import optuna
from sklearn.impute import SimpleImputer
import os
from imblearn.pipeline import Pipeline 
import xgboost as  xgb
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, precision_score
import sys
from sklearn.experimental import enable_iterative_imputer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import joblib
import logging
from sklearn.model_selection import train_test_split
import optuna
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.svm import SVC
import lightgbm as lgb
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings("ignore")




ruta_raiz = os.path.abspath('..')
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

%load_ext autoreload
%autoreload 2
from Data_preprocesing.IQRCapper import IQRCapper



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LOS OUTPUTS DE OPTUNA QUE LOS IMPRIMA EN UN TXT PARA SIN VOLVERLO A CORRER

In [3]:
log_file = logging.FileHandler("optuna_resultados.txt")
log_file.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
optuna.logging.get_logger("optuna").addHandler(log_file)

In [ ]:
TRAIN = False
SAVE_MODEL = False
CHARGE_XBOOST= False
CHOOSE_SAMPLER = False

## LOAD DATA + SAVE FUCTION


In [41]:
X_train = pd.read_csv("../Train_test_split/X_train.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv")
X_test = pd.read_csv("../Train_test_split/X_test.csv")
y_test = pd.read_csv("../Train_test_split/y_test.csv")



target_folder = '../comparations'

def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    # Point the file to inside the folder
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")


## XGBOOST

In [6]:
num_classes = int(y_train['customer_segment'].nunique())
modelo_xgb = xgb.XGBClassifier(
    objective='multi:softmax', 
    num_class=num_classes, 
    random_state=42,
    eval_metric='mlogloss'
)

## DEFINIMOS LOS RANGOS Y CATEGORIAS QUE QUEREMOS QUE OPTUNA OPTIMICE

In [42]:
if CHOOSE_SAMPLER:
    imbalance_methods = {
        "Baseline": None,
        "RandomUnderSampler":     RandomUnderSampler(random_state=42),
        "TomekLinks":             TomekLinks(),
        "RandomOverSampler":      RandomOverSampler(random_state=42),
        "SMOTE":                  SMOTE(random_state=42),
        "ADASYN":                 ADASYN(random_state=42),
        "SMOTETomek":             SMOTETomek(random_state=42),
        "SMOTEENN":               SMOTEENN(random_state=42),
    }


    for method_name, sampler in imbalance_methods.items():
        print(f"\n--- {method_name} ---")

        

        if sampler is None:
            pipe = Pipeline([
                ('capping_outliers', IQRCapper(factor=1.5)),
                ('imputacion', SimpleImputer(strategy='mean')),
                ('scaler', StandardScaler()),
                ('classifier', modelo_xgb)])
        else:
            pipe = Pipeline([
                ('capping_outliers', IQRCapper(factor=1.5)),
                ('imputacion', SimpleImputer(strategy='mean')),
                ('scaler', StandardScaler()),
                ('sampler', sampler), 
                ('classifier', modelo_xgb)])

        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)

        accuracy  = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

        print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

        save_experiment(
            model_name="XgBoost",
            imbalance_method=method_name,
            accuracy=accuracy,
            precision=precision,
            recall=recall,
            f1=f1,
            roc_auc=roc_auc
        )


--- Baseline ---
Accuracy: 0.7531 | F1: 0.7517 | ROC-AUC: 0.9009
✅ Success! Results for XgBoost + Baseline saved in the '../comparations' folder.

--- RandomUnderSampler ---
Accuracy: 0.7276 | F1: 0.7313 | ROC-AUC: 0.8928
✅ Success! Results for XgBoost + RandomUnderSampler saved in the '../comparations' folder.

--- TomekLinks ---
Accuracy: 0.7564 | F1: 0.7545 | ROC-AUC: 0.9018
✅ Success! Results for XgBoost + TomekLinks saved in the '../comparations' folder.

--- RandomOverSampler ---
Accuracy: 0.7425 | F1: 0.7455 | ROC-AUC: 0.8988
✅ Success! Results for XgBoost + RandomOverSampler saved in the '../comparations' folder.

--- SMOTE ---
Accuracy: 0.7501 | F1: 0.7512 | ROC-AUC: 0.8989
✅ Success! Results for XgBoost + SMOTE saved in the '../comparations' folder.

--- ADASYN ---
Accuracy: 0.7442 | F1: 0.7448 | ROC-AUC: 0.8983
✅ Success! Results for XgBoost + ADASYN saved in the '../comparations' folder.

--- SMOTETomek ---
Accuracy: 0.7509 | F1: 0.7521 | ROC-AUC: 0.8992
✅ Success! Results

In [ ]:
def objective(trial):
    

    # ESCOGER TIPO DE MUESTREO
    nombre_metodo = "TomekLinks" 
    metodo_elegido = TomekLinks()
    # ESCOGER LOS HIPERPARÁMETROS DE XGBOOST
    param_n_estimators = trial.suggest_int('n_estimators', 50, 1000)
    param_max_depth = trial.suggest_int('max_depth', 3, 8)  
    param_learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3, log=True)
    param_gamma = trial.suggest_float('gamma', 0.0, 5.0 )
    param_lambda = trial.suggest_float('lambda', 0.5, 10.0, log=True) 
    param_alpha = trial.suggest_float('alpha', 0.0, 5.0)
    param_subsample = trial.suggest_float('subsample',        0.5, 1.0)
    param_colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    param_min_child_weight = trial.suggest_int(  'min_child_weight', 1, 20)  # 
    param_max_delta_step = trial.suggest_float('max_delta_step', 0.0, 1.0)
    param_iqr_factor = trial.suggest_float('iqr_factor', 1.0, 3.0)



    modelo_xgb = xgb.XGBClassifier(
        n_estimators = param_n_estimators,
        max_depth = param_max_depth,
        learning_rate = param_learning_rate,
        gamma = param_gamma,
        reg_lambda = param_lambda,
        reg_alpha = param_alpha,
        subsample = param_subsample,
        colsample_bytree = param_colsample_bytree,
        min_child_weight = param_min_child_weight,
        max_delta_step = param_max_delta_step,
        objective = 'multi:softmax',
        eval_metric = 'mlogloss',
        num_class = len(np.unique(y_train)),
        random_state = 42,
        n_jobs = -1
    )

    pipeline = Pipeline([
        ('capping_outliers', IQRCapper(factor=param_iqr_factor)),
        ('imputacion',       SimpleImputer(strategy='mean')),
        ('scaler',           StandardScaler()),
        ('sampler', metodo_elegido),
        ('clf',              modelo_xgb)
    ])

    
    sample_weights = compute_sample_weight('balanced', y_train)


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    scoring = 'roc_auc_ovr_weighted',
    cv=5,
    n_jobs=1)

    return np.mean(scores)



if TRAIN:
    best_value = [None]
    no_improve_count = [0]

    def early_stopping_callback(study, trial):
        if best_value[0] is None or study.best_value > best_value[0]:
            best_value[0] = study.best_value
            no_improve_count[0] = 0
        else:
            no_improve_count[0] += 1
        if no_improve_count[0] >= 50:
            study.stop()

    study = optuna.create_study(direction="maximize", 
                                sampler=optuna.samplers.TPESampler(seed=42),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)  
)
    study.optimize(objective, n_trials=200, callbacks=[early_stopping_callback], show_progress_bar = True)
    print(f"Mejor F1-Score Macro: {study.best_value:.4f}")
    print("Mejor combinación encontrada:")
    for key, value in study.best_params.items():
        print(f"  - {key}: {value}")

    with open("mejores_parametros_XGBoost.txt", "w") as archivo:
        archivo.write("Mejores hiperparámetros encontrados para XGBoost:\n")
        archivo.write("="*40 + "\n")
        for key, value in study.best_params.items():
            archivo.write(f"  - {key}: {value}\n")

    print("¡Parámetros guardados correctamente en 'mejores_parametros.txt'!")
    
            
  

GUARDAMOS EL MEJOR MODELO

In [ ]:
if TRAIN:
    print("Reconstruyendo y entrenando el modelo final con toda la data...")
    
    mejor_metodo_nombre = "TomekLinks"
    mejor_metodo = TomekLinks()
    sample_weights = compute_sample_weight('balanced', y_train)

    mejor_xgb = xgb.XGBClassifier(
        n_estimators=study.best_params['n_estimators'],
        max_depth=study.best_params['max_depth'],
        learning_rate=study.best_params['learning_rate'],
        subsample=study.best_params['subsample'],
        colsample_bytree=study.best_params['colsample_bytree'],
        objective = 'multi:softprob',
        eval_metric='mlogloss',
        tree_method = 'hist', 
        random_state=42,
        n_jobs=-1
    )

    pasos_finales = [
        ('capping_outliers', IQRCapper(factor=study.best_params["iqr_factor"])),
        ('imputacion', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('sampler', mejor_metodo),
        ('clf', mejor_xgb)
    ]



    pipeline_final = Pipeline(pasos_finales)


if SAVE_MODEL:
    joblib.dump(pipeline_final, 'modelo_xgboost_ganador.pkl')
    pipeline_final.fit(X_train, y_train)
    if SAVE_MODEL:
        joblib.dump(pasos_finales, 'modelo_xgboost_ganador.pkl')
        print("¡Modelo final entrenado y guardado correctamente como 'modelo_xgboost_ganador.pkl'!")
        


In [ ]:
if CHARGE_XBOOST:
    pasos_cargados = joblib.load('modelo_xgboost_ganador.pkl')
    pipeline = Pipeline(pasos_cargados)

    X_train = pd.read_csv("../Train_test_split/X_train.csv")
    y_train = pd.read_csv("../Train_test_split/y_train.csv")
    X_test = pd.read_csv("../Train_test_split/X_test.csv")
    y_test = pd.read_csv("../Train_test_split/y_test.csv")

    y_pred       = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba, 
                            multi_class='ovr',  
                            average='weighted')  

    new_result = {
        'Model': ["XGBoost"],
        'Imbalance_Method': [pasos_cargados[3][1]],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }

    print(new_result)



## STACKING CLASIFFIER

In [ ]:
TRAINSTACK =False
SAVESTACK = False
CHOOSESAMPLER = False
CHARGE_STACKING = False
SAVE_MODEL_STACK = False

In [44]:
imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}

# ============================================================
# FASE 1: Selección del mejor sampler con LightGBM (proxy rápido)
# ============================================================
if CHOOSESAMPLER:
    best_sampler_name = None
    best_f1 = 0
    quick_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

    for method_name, sampler in imbalance_methods.items():
        steps = [
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
        ]
        if sampler is not None:
            steps.append(('sampler', sampler))
        steps.append(('classifier', quick_model))

        pipe = Pipeline(steps)
        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)

        accuracy  = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

        print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

        save_experiment(
            model_name="Stacking Clafier",
            imbalance_method=method_name,
            accuracy=accuracy,
            precision=precision,
            recall=recall,
            f1=f1,
            roc_auc=roc_auc
        )
        print(f"{method_name}: F1={f1:.4f}")
        if method_name == "Baseline": f1+=0.01   #Vamos a dar una pequeña ventaja a lo simple
        if f1 > best_f1:
            best_f1          = f1
            best_sampler_name = method_name

    print(f"\nMejor sampler: {best_sampler_name} (F1={best_f1:.4f})") 
best_sampler_name = "Baseline"
# ============================================================
# FASE 2: Stacking final solo con el mejor sampler
# ============================================================
modelos_base = [
    ('xgb', xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
        n_estimators=100,
        max_depth=4,
        tree_method='hist',
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )),
    ('svm', CalibratedClassifierCV(
        LinearSVC(random_state=42, max_iter=2000),
        cv=3
    )),
    ('lgb', lgb.LGBMClassifier(random_state=42, n_jobs=-1))
]

stacking_clf = StackingClassifier(
    estimators=modelos_base,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

best_sampler = imbalance_methods["Baseline"]

steps_final = [
    ('capping_outliers', IQRCapper(factor=1.5)),
    ('imputacion', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
]
if best_sampler is not None:
    steps_final.append(('sampler', best_sampler))
steps_final.append(('classifier', stacking_clf))

pipe_final = Pipeline(steps_final)
pipe_final.fit(X_train, y_train)

y_pred  = pipe_final.predict(X_test)
y_proba = pipe_final.predict_proba(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print(f"\n--- Stacking Final ({best_sampler_name}) ---")
print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")


"""
Entrenar el Stacking Classifier completo con cada uno de los 8 métodos de balanceo es computacionalmente inviable, 
ya que cada iteración implica reentrenar múltiples modelos costosos con validación cruzada. 
Para reducir el coste sin sacrificar rigor, se adopta una estrategia en dos fases: primero se selecciona el método de balanceo óptimo usando 
LightGBM como modelo proxy —rápido, robusto y habitualmente correlacionado en rendimiento con ensambles más complejos—, y 
posteriormente se entrena el Stacking únicamente con dicho método. Esto reduce el número de entrenamientos costosos de 8 a 1, manteniendo una selección informada del sampler.
"""

Accuracy: 0.7611 | F1: 0.7597 | ROC-AUC: 0.9053
✅ Success! Results for Stacking Clafier + Baseline saved in the '../comparations' folder.
Baseline: F1=0.7597
Accuracy: 0.7402 | F1: 0.7436 | ROC-AUC: 0.8988
✅ Success! Results for Stacking Clafier + RandomUnderSampler saved in the '../comparations' folder.
RandomUnderSampler: F1=0.7436
Accuracy: 0.7615 | F1: 0.7597 | ROC-AUC: 0.9048
✅ Success! Results for Stacking Clafier + TomekLinks saved in the '../comparations' folder.
TomekLinks: F1=0.7597
Accuracy: 0.7487 | F1: 0.7521 | ROC-AUC: 0.9042
✅ Success! Results for Stacking Clafier + RandomOverSampler saved in the '../comparations' folder.
RandomOverSampler: F1=0.7521
Accuracy: 0.7583 | F1: 0.7596 | ROC-AUC: 0.9032
✅ Success! Results for Stacking Clafier + SMOTE saved in the '../comparations' folder.
SMOTE: F1=0.7596
Accuracy: 0.7475 | F1: 0.7484 | ROC-AUC: 0.9030
✅ Success! Results for Stacking Clafier + ADASYN saved in the '../comparations' folder.
ADASYN: F1=0.7484
Accuracy: 0.7561 | F

'\nEntrenar el Stacking Classifier completo con cada uno de los 8 métodos de balanceo es computacionalmente inviable, \nya que cada iteración implica reentrenar múltiples modelos costosos con validación cruzada. \nPara reducir el coste sin sacrificar rigor, se adopta una estrategia en dos fases: primero se selecciona el método de balanceo óptimo usando \nLightGBM como modelo proxy —rápido, robusto y habitualmente correlacionado en rendimiento con ensambles más complejos—, y \nposteriormente se entrena el Stacking únicamente con dicho método. Esto reduce el número de entrenamientos costosos de 8 a 1, manteniendo una selección informada del sampler.\n'

In [23]:
if TRAINSTACK:
    def objective_stacking(trial):

        X_opt, _, y_opt, _ = train_test_split(
            X_train, y_train,
            train_size=10_000,
            stratify=y_train,
            random_state=42
)
        # ── META-MODELO ──────────────────────────────────────────────────
        meta_C = trial.suggest_float('meta_C', 1e-3, 10.0, log=True)
        meta_model = LogisticRegression(C=meta_C, max_iter=1000, random_state=42)

        # ── BASE LEARNER 1: XGBoost ───────────────────────────────────────
        xgb_model = xgb.XGBClassifier(
            n_estimators = trial.suggest_int('xgb_n_estimators', 50, 500),
            max_depth = trial.suggest_int('xgb_max_depth',3, 8),
            learning_rate = trial.suggest_float('xgb_learning_rate', 0.005, 0.3, log=True),
            subsample = trial.suggest_float('xgb_subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('xgb_colsample_bytree',0.5, 1.0),
            gamma = trial.suggest_float('xgb_gamma', 0.0, 5.0),
            reg_lambda = trial.suggest_float('xgb_lambda', 0.5, 10.0, log=True),
            reg_alpha = trial.suggest_float('xgb_alpha', 0.0, 5.0),
            min_child_weight = trial.suggest_int('xgb_min_child_weight', 1, 20),
            objective = 'multi:softprob',
            eval_metric = 'mlogloss',
            num_class = len(np.unique(y_train)),
            tree_method = 'hist',
            random_state = 42,
            n_jobs = -1
        )

        # ── BASE LEARNER 2: Random Forest ────────────────────────────────
        rf_model = RandomForestClassifier(
            n_estimators = trial.suggest_int('rf_n_estimators', 50, 500),
            max_depth = trial.suggest_int('rf_max_depth', 3, 20),
            max_features = trial.suggest_float('rf_max_features', 0.3, 1.0),
            min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 20),
            min_samples_leaf  = trial.suggest_int('rf_min_samples_leaf', 1, 10),
            random_state = 42,
            n_jobs = -1
        )

        # ── BASE LEARNER 3: SVM (rbf, con probabilidades) ────────────────
        svm_model = SVC(
            C = trial.suggest_float('svm_C', 1e-2, 100.0, log=True),
            gamma = trial.suggest_float('svm_gamma', 1e-4, 1.0, log=True),
            kernel = 'rbf',
            probability = True,
            random_state= 42
        )

        # ── BASE LEARNER 4: LightGBM ─────────────────────────────────────
        lgbm_model = lgb.LGBMClassifier(
            n_estimators  = trial.suggest_int('lgbm_n_estimators', 50, 500),
            max_depth = trial.suggest_int('lgbm_max_depth', 3, 8),
            learning_rate = trial.suggest_float('lgbm_learning_rate', 0.005, 0.3, log=True),
            num_leaves = trial.suggest_int('lgbm_num_leaves', 15, 127),
            subsample = trial.suggest_float('lgbm_subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('lgbm_colsample_bytree', 0.5, 1.0),
            reg_alpha = trial.suggest_float('lgbm_alpha', 0.0, 5.0),
            reg_lambda = trial.suggest_float('lgbm_lambda', 0.5, 10.0, log=True),
            objective = 'multiclass',
            random_state  = 42,
            n_jobs = -1,
            verbose = -1
        )

        # ── STACKING ─────────────────────────────────────────────────────
        stacking = StackingClassifier(
            estimators=[
                ('xgb',xgb_model),
                ('rf', rf_model),
                ('svm',svm_model),
                ('lgbm', lgbm_model),
            ],
            final_estimator = meta_model,
            cv = 5,
            stack_method = 'predict_proba',
            n_jobs = -1
        )

        # ── PIPELINE ─────────────────────────────────────────────────────
        steps = [
            ('capping_outliers', IQRCapper(factor=trial.suggest_float('iqr_factor', 1.0, 3.0))),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
        ]
        if best_sampler is not None:
            steps.append(('sampler', best_sampler))
        steps.append(('classifier', stacking))

        pipeline = Pipeline(steps)

        cv_splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        sample_weights = compute_sample_weight('balanced', y_train)

        scores = cross_val_score(
            pipeline,
            X_opt, y_opt,       # ← solo para buscar hiperparámetros
            cv      = cv_splitter,
            scoring = 'roc_auc_ovr_weighted',
            n_jobs  = 1
        )

        return np.mean(scores)


# ── EJECUCIÓN ─────────────────────────────────────────────────────────
if TRAINSTACK:
    best_value = [None]
    no_improve_count = [0]

    
    study = optuna.create_study(
        direction = "maximize",
        sampler = optuna.samplers.TPESampler(seed=42),
        pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
    )

    study.optimize(
        objective_stacking,
        n_trials = 20,
        callbacks = [early_stopping_callback],
        show_progress_bar = True
    )

    print(f"\nMejor ROC-AUC (weighted OvR): {study.best_value:.4f}")
    print("Mejores hiperparámetros:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

    with open("mejores_parametros_Stacking.txt", "w") as f:
        f.write("Mejores hiperparámetros — Stacking Classifier\n")
        f.write("=" * 45 + "\n")
        for k, v in study.best_params.items():
            f.write(f"  {k}: {v}\n")

    print("\n¡Parámetros guardados en 'mejores_parametros_Stacking.txt'!")

[I 2026-03-16 19:56:00,889] A new study created in memory with name: no-name-34b8bcec-a723-40c8-9479-7c7120fa4b22
Best trial: 0. Best value: 0.896575:   5%|▌         | 1/20 [01:47<34:07, 107.75s/it]

[I 2026-03-16 19:57:48,634] Trial 0 finished with value: 0.8965746784633972 and parameters: {'meta_C': 0.03148911647956861, 'xgb_n_estimators': 478, 'xgb_max_depth': 7, 'xgb_learning_rate': 0.058006322999333615, 'xgb_subsample': 0.5780093202212182, 'xgb_colsample_bytree': 0.5779972601681014, 'xgb_gamma': 0.2904180608409973, 'xgb_lambda': 6.697167353375242, 'xgb_alpha': 3.005575058716044, 'xgb_min_child_weight': 15, 'rf_n_estimators': 59, 'rf_max_depth': 20, 'rf_max_features': 0.8827098485602951, 'rf_min_samples_split': 6, 'rf_min_samples_leaf': 2, 'svm_C': 0.05415244119402541, 'svm_gamma': 0.0016480446427978971, 'lgbm_n_estimators': 286, 'lgbm_max_depth': 5, 'lgbm_learning_rate': 0.01647477394109053, 'lgbm_num_leaves': 84, 'lgbm_subsample': 0.569746930326021, 'lgbm_colsample_bytree': 0.6460723242676091, 'lgbm_alpha': 1.8318092164684585, 'lgbm_lambda': 1.9603369861210684, 'iqr_factor': 2.570351922786027}. Best is trial 0 with value: 0.8965746784633972.


Best trial: 0. Best value: 0.896575:  10%|█         | 2/20 [03:33<31:56, 106.50s/it]

[I 2026-03-16 19:59:34,259] Trial 1 finished with value: 0.892803330445818 and parameters: {'meta_C': 0.006290644294586149, 'xgb_n_estimators': 281, 'xgb_max_depth': 6, 'xgb_learning_rate': 0.006047360568422396, 'xgb_subsample': 0.8037724259507192, 'xgb_colsample_bytree': 0.5852620618436457, 'xgb_gamma': 0.3252579649263976, 'xgb_lambda': 8.58022251487741, 'xgb_alpha': 4.828160165372797, 'xgb_min_child_weight': 17, 'rf_n_estimators': 187, 'rf_max_depth': 4, 'rf_max_features': 0.7789631185585097, 'rf_min_samples_split': 10, 'rf_min_samples_leaf': 2, 'svm_C': 0.9565499215943827, 'svm_gamma': 0.00013726318898045882, 'lgbm_n_estimators': 460, 'lgbm_max_depth': 4, 'lgbm_learning_rate': 0.07534159891754702, 'lgbm_num_leaves': 50, 'lgbm_subsample': 0.7600340105889054, 'lgbm_colsample_bytree': 0.7733551396716398, 'lgbm_alpha': 0.9242722776276352, 'lgbm_lambda': 9.129115219600122, 'iqr_factor': 2.550265646722229}. Best is trial 0 with value: 0.8965746784633972.


Best trial: 2. Best value: 0.898715:  15%|█▌        | 3/20 [05:41<33:02, 116.59s/it]

...

[I 2026-03-16 20:33:01,392] Trial 17 finished with value: 0.8988280603978905 and parameters: {'meta_C': 0.14043608665217464, 'xgb_n_estimators': 213, 'xgb_max_depth': 3, 'xgb_learning_rate': 0.1487975750758097, 'xgb_subsample': 0.8914253704410535, 'xgb_colsample_bytree': 0.9140229556557437, 'xgb_gamma': 0.9629206289988517, 'xgb_lambda': 1.8205700160289777, 'xgb_alpha': 2.481165187056286, 'xgb_min_child_weight': 12, 'rf_n_estimators': 395, 'rf_max_depth': 9, 'rf_max_features': 0.9987985103162023, 'rf_min_samples_split': 17, 'rf_min_samples_leaf': 10, 'svm_C': 4.9022168209785875, 'svm_gamma': 0.004015514744490561, 'lgbm_n_estimators': 332, 'lgbm_max_depth': 3, 'lgbm_learning_rate': 0.10901514051073383, 'lgbm_num_leaves': 125, 'lgbm_subsample': 0.5120388064366901, 'lgbm_colsample_bytree': 0.6263478194252343, 'lgbm_alpha': 2.146505297567076, 'lgbm_lambda': 0.7699904610193301, 'iqr_factor': 2.9393565551002414}. Best is trial 14 with value: 0.8999946687747334.


Best trial: 14. Best value: 0.899995:  95%|█████████▌| 19/20 [38:33<01:46, 106.47s/it]

[I 2026-03-16 20:34:34,749] Trial 18 finished with value: 0.8980804388660985 and parameters: {'meta_C': 0.111008517909964, 'xgb_n_estimators': 334, 'xgb_max_depth': 5, 'xgb_learning_rate': 0.010983876875834582, 'xgb_subsample': 0.8588837623812291, 'xgb_colsample_bytree': 0.8973680390720666, 'xgb_gamma': 2.1568913212602996, 'xgb_lambda': 4.021436078533608, 'xgb_alpha': 2.5126691033654027, 'xgb_min_child_weight': 10, 'rf_n_estimators': 287, 'rf_max_depth': 9, 'rf_max_features': 0.5824133238557088, 'rf_min_samples_split': 17, 'rf_min_samples_leaf': 9, 'svm_C': 0.49372764179218304, 'svm_gamma': 0.03587377556080421, 'lgbm_n_estimators': 331, 'lgbm_max_depth': 4, 'lgbm_learning_rate': 0.0415987044649615, 'lgbm_num_leaves': 108, 'lgbm_subsample': 0.6164168514029683, 'lgbm_colsample_bytree': 0.7089846258138215, 'lgbm_alpha': 1.2371419678536455, 'lgbm_lambda': 0.9569665498851504, 'iqr_factor': 2.2779216160412785}. Best is trial 14 with value: 0.8999946687747334.


Best trial: 14. Best value: 0.899995: 100%|██████████| 20/20 [40:25<00:00, 121.28s/it]

[I 2026-03-16 20:36:26,493] Trial 19 finished with value: 0.8962547523204831 and parameters: {'meta_C': 0.010462545669018075, 'xgb_n_estimators': 228, 'xgb_max_depth': 3, 'xgb_learning_rate': 0.038101263573073474, 'xgb_subsample': 0.7490722248603884, 'xgb_colsample_bytree': 0.9996743898633776, 'xgb_gamma': 1.1761000772985324, 'xgb_lambda': 0.5920057947222835, 'xgb_alpha': 1.0660137107798846, 'xgb_min_child_weight': 3, 'rf_n_estimators': 297, 'rf_max_depth': 12, 'rf_max_features': 0.6968276754988628, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 9, 'svm_C': 2.75824351426793, 'svm_gamma': 0.0004328834810223492, 'lgbm_n_estimators': 401, 'lgbm_max_depth': 4, 'lgbm_learning_rate': 0.20148184687797893, 'lgbm_num_leaves': 127, 'lgbm_subsample': 0.5212015197443293, 'lgbm_colsample_bytree': 0.6008470664371457, 'lgbm_alpha': 2.8562232627455666, 'lgbm_lambda': 3.8965716918897715, 'iqr_factor': 2.607778492705731}. Best is trial 14 with value: 0.8999946687747334.

Mejor ROC-AUC (weighted OvR)

In [33]:
best =  {
    # Meta
    "meta_C": 0.26322176445064177,

    # XGBoost
    "xgb_n_estimators": 335,
    "xgb_max_depth": 4,
    "xgb_learning_rate": 0.1446304576052762,
    "xgb_subsample": 0.8810754397500585,
    "xgb_colsample_bytree": 0.863339044684348,
    "xgb_gamma": 2.1774582834198766,
    "xgb_lambda": 1.8517526387199372,
    "xgb_alpha": 0.9541790020797591,
    "xgb_min_child_weight": 18,

    # Random Forest
    "rf_n_estimators": 365,
    "rf_max_depth": 7,
    "rf_max_features": 0.555690367550149,
    "rf_min_samples_split": 15,
    "rf_min_samples_leaf": 8,

    # SVM
    "svm_C": 4.702572566470305,
    "svm_gamma": 0.0049969091056217205,

    # LightGBM
    "lgbm_n_estimators": 369,
    "lgbm_max_depth": 6,
    "lgbm_learning_rate": 0.009393650881828619,
    "lgbm_num_leaves": 101,
    "lgbm_subsample": 0.6795264245309642,
    "lgbm_colsample_bytree": 0.6909998058953244,
    "lgbm_alpha": 0.8429078529399413,
    "lgbm_lambda": 0.7702736000283799,

    # Outlier removal
    "iqr_factor": 2.335117092184226,
}
if SAVE_MODEL_STACK:
    print("Reconstruyendo y entrenando el modelo final con toda la data...")

    best = study.best_params

    # ── META-MODELO ──────────────────────────────────────────────────
    meta_model = LogisticRegression(C=best['meta_C'], max_iter=1000, random_state=42)

    # ── BASE LEARNER 1: XGBoost ──────────────────────────────────────
    xgb_model = xgb.XGBClassifier(
        n_estimators = best['xgb_n_estimators'],
        max_depth = best['xgb_max_depth'],
        learning_rate = best['xgb_learning_rate'],
        subsample = best['xgb_subsample'],
        colsample_bytree = best['xgb_colsample_bytree'],
        gamma = best['xgb_gamma'],
        reg_lambda = best['xgb_lambda'],
        reg_alpha = best['xgb_alpha'],
        min_child_weight = best['xgb_min_child_weight'],
        objective = 'multi:softprob',
        eval_metric = 'mlogloss',
        num_class = len(np.unique(y_train)),
        tree_method = 'hist',
        random_state = 42,
        n_jobs  = -1
    )

    # ── BASE LEARNER 2: Random Forest ────────────────────────────────
    rf_model = RandomForestClassifier(
        n_estimators = best['rf_n_estimators'],
        max_depth = best['rf_max_depth'],
        max_features = best['rf_max_features'],
        min_samples_split = best['rf_min_samples_split'],
        min_samples_leaf = best['rf_min_samples_leaf'],
        random_state = 42,
        n_jobs= -1
    )

    # ── BASE LEARNER 3: SVM ──────────────────────────────────────────
    svm_model = SVC(
        C = best['svm_C'],
        gamma = best['svm_gamma'],
        kernel = 'rbf',
        probability = True,
        random_state = 42
    )

    # ── BASE LEARNER 4: LightGBM ─────────────────────────────────────
    lgbm_model = lgb.LGBMClassifier(
        n_estimators = best['lgbm_n_estimators'],
        max_depth = best['lgbm_max_depth'],
        learning_rate = best['lgbm_learning_rate'],
        num_leaves = best['lgbm_num_leaves'],
        subsample = best['lgbm_subsample'],
        colsample_bytree = best['lgbm_colsample_bytree'],
        reg_alpha = best['lgbm_alpha'],
        reg_lambda = best['lgbm_lambda'],
        objective = 'multiclass',
        random_state = 42,
        n_jobs = -1,
        verbose = -1
    )

    # ── STACKING ─────────────────────────────────────────────────────
    stacking_final = StackingClassifier(
        estimators=[
            ('xgb', xgb_model),
            ('rf', rf_model),
            ('svm', svm_model),
            ('lgbm', lgbm_model),
        ],
        final_estimator = meta_model,
        cv = 5,
        stack_method = 'predict_proba',
        n_jobs = -1
    )

    # ── PIPELINE FINAL ───────────────────────────────────────────────
    steps_final = [
        ('capping_outliers', IQRCapper(factor=best['iqr_factor'])),
        ('imputacion', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
    ]
    if best_sampler is not None:
        steps_final.append(('sampler', best_sampler))
    steps_final.append(('classifier', stacking_final))

    pipeline_final = Pipeline(steps_final)

    # Sin sample_weight: SVC dentro del stacking no lo propaga correctamente
    pipeline_final.fit(X_train, y_train)

    joblib.dump(pipeline_final, 'modelo_stacking_ganador.pkl')
    print("¡Modelo final entrenado y guardado correctamente como 'modelo_stacking_ganador.pkl'!")

Reconstruyendo y entrenando el modelo final con toda la data...
¡Modelo final entrenado y guardado correctamente como 'modelo_stacking_ganador.pkl'!


In [34]:
if CHARGE_STACKING:
    pipeline_final = joblib.load('modelo_stacking_ganador.pkl')

    X_train = pd.read_csv("../Train_test_split/X_train.csv")
    y_train = pd.read_csv("../Train_test_split/y_train.csv")
    X_test = pd.read_csv("../Train_test_split/X_test.csv")
    y_test = pd.read_csv("../Train_test_split/y_test.csv")

    y_pred = pipeline_final.predict(X_test)
    y_pred_proba = pipeline_final.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba,
                              multi_class='ovr',
                              average='weighted')

    new_result = {
        'Model': ["Stacking Classifier"],
        'Imbalance_Method': [best_sampler_name],
        'Accuracy': [round(accuracy,  4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall,    4)],
        'F1_Score': [round(f1,        4)],
        'ROC_AUC': [round(roc_auc,   4)]
    }

    print(new_result)

{'Model': ['Stacking Classifier'], 'Imbalance_Method': ['Baseline'], 'Accuracy': [0.7635], 'Precision': [0.7621], 'Recall': [0.7635], 'F1_Score': [0.7616], 'ROC_AUC': [0.9072]}
